# N2 solution. Guided activity of Notebook 2, solved

**Predicting Flash Floods in Working Landscapes**, Module 3, Event based evaluation and forecast skill
CUAHSI Water Science Exchange 2026

This short notebook holds the completed activity of Notebook 2 (section 6), with its outputs. It rebuilds only what the activity needs, so it runs on its own in a few seconds. The full notebook with every section executed is `02_ensemble_to_probability_solution.ipynb`.

The task. How many members does a probability need? Repeat the probability of exceedance with the first 20 members only and compare it with the 100 member version.

## 0. Setup

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AHWALab/Water-Science-Exchange-Workshop.git"
BRANCH = "module-3-evaluation"

# In Colab the helper files are not here yet. Copy the branch (public, no account needed).
if not Path("hymod_tools.py").exists():
    if not Path("Water-Science-Exchange-Workshop").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "-b", BRANCH, REPO_URL, "Water-Science-Exchange-Workshop"], check=True)
    os.chdir("Water-Science-Exchange-Workshop/notebooks")
sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())

Working folder: /home/claude/m3/notebooks


## 1. What the activity needs from Notebook 2

The data, the flood threshold, the 100 member ensemble (same seed, so the identical ensemble) and the 100 member probability of exceedance of section 4.

In [2]:
import numpy as np
import hymod_tools as h

PP, PET, QQ = h.load_leaf_river("LeafCatch.dat")      # 40 years of daily data, mm per day
N_EVAL = 365 * 5                                        # the five year evaluation window
Qobs = QQ[:N_EVAL]
flood_th = h.flood_threshold(QQ, fraction=0.25)         # 5.0 mm per day
ensemble, members = h.generate_ensemble(PP, PET, N_EVAL, Nens=100, seed=7)
prob_exceed = (ensemble >= flood_th).mean(axis=0)       # share of members at or above the threshold, each day
print(f"ensemble {ensemble.shape}, threshold {flood_th:.1f} mm/day, largest probability {prob_exceed.max():.2f}")

ensemble (100, 1825), threshold 5.0 mm/day, largest probability 0.15


## 2. The activity, solved

In [3]:
#################### Your Code Goes Here #########################
# Step 1. Keep the first 20 members of the ensemble (rows of the array).
small = ensemble[:20]

# Step 2. Probability of exceedance from those 20 members, same recipe as section 4.
prob_small = (small >= flood_th).mean(axis=0)

# Step 3. Compare with prob_exceed. Print the largest absolute difference on any day
#         and the correlation between the two series (np.corrcoef).
print(f"largest difference on a day {np.abs(prob_small - prob_exceed).max():.2f}")
print(f"correlation between the two series {np.corrcoef(prob_small, prob_exceed)[0, 1]:.3f}")
###################################################################

largest difference on a day 0.09
correlation between the two series 0.875


Reading the result. With 20 members the probability can only take the values 0, 0.05, 0.10 and so on, so rare events get coarse probabilities, and on single days the two versions can differ by several hundredths. Over the whole window the two series are still highly correlated, which is why operational ensembles of 20 to 50 members remain useful.